# Unpack ImageCLEF MEDVQA GI 2023 dataset zips

Place the two zip files in `0_dataset_prep/dataset_download_zip` (or set `AUTO_CONFIG` with explicit paths) and run this notebook.
It will create the following folders under `out/`:

- `ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset`
- `ImageCLEFmed-MEDVQA-GI-2023-Testing-Dataset`

If the target folder already exists, extraction is skipped unless `force=True`.


In [3]:
from pathlib import Path
import os
import zipfile
import shutil

AUTO_CONFIG = globals().get("AUTO_CONFIG", {
    # "dev_zip": "/path/to/ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset.zip",
    # "test_zip": "/path/to/ImageCLEFmed-MEDVQA-GI-2023-Testing-Dataset.zip",
    # "out_dir": "/path/to/0_dataset_prep/out",
    # "zip_dir": "/path/to/0_dataset_prep/dataset_download_zip",
    # "force": False,
})

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
OUT_DIR = Path(AUTO_CONFIG.get("out_dir", ROOT / "0_dataset_prep" / "out")).expanduser().resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DIR = Path(AUTO_CONFIG.get("zip_dir", ROOT / "0_dataset_prep" / "dataset_download_zip")).expanduser().resolve()
ZIP_DIR.mkdir(parents=True, exist_ok=True)

DEV_NAME = "ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset"
TEST_NAME = "ImageCLEFmed-MEDVQA-GI-2023-Testing-Dataset"

def _find_zip(dataset_name: str, config_key: str) -> Path | None:
    cfg = AUTO_CONFIG.get(config_key)
    if cfg:
        p = Path(cfg).expanduser().resolve()
        if p.exists():
            return p
        raise FileNotFoundError(f"{config_key} points to missing file: {p}")

    candidates = []
    search_dirs = [ZIP_DIR, OUT_DIR]
    for d in search_dirs:
        if not d.exists():
            continue
        candidates += [
            p for p in d.iterdir()
            if p.is_file() and p.suffix.lower() == ".zip" and dataset_name.lower() in p.name.lower()
        ]
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple zip candidates for {dataset_name}: {candidates}")
    return None

DEV_ZIP = _find_zip(DEV_NAME, "dev_zip")
TEST_ZIP = _find_zip(TEST_NAME, "test_zip")
FORCE = bool(AUTO_CONFIG.get("force", False))

print(f"ROOT: {ROOT}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"ZIP_DIR: {ZIP_DIR}")
print(f"DEV_ZIP: {DEV_ZIP}")
print(f"TEST_ZIP: {TEST_ZIP}")


ROOT: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023
OUT_DIR: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/out
ZIP_DIR: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/dataset_download_zip
DEV_ZIP: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/dataset_download_zip/ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset.zip
TEST_ZIP: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/dataset_download_zip/ImageCLEFmed-MEDVQA-GI-2023-Testing-Dataset.zip


In [4]:
def _iter_members(zf: zipfile.ZipFile):
    for info in zf.infolist():
        name = info.filename.replace("\\", "/")
        if not name or name.endswith("/"):
            continue
        if name.startswith("__MACOSX/") or "/__MACOSX/" in name:
            continue
        if name.endswith(".DS_Store"):
            continue
        yield info, name


def _safe_extract_member(zf: zipfile.ZipFile, info: zipfile.ZipInfo, dest_root: Path, rel: str) -> Path:
    rel_path = Path(rel)
    if rel_path.is_absolute():
        raise RuntimeError(f"Refusing to extract absolute path: {rel}")

    dest_root_resolved = dest_root.resolve()
    dest = (dest_root / rel_path).resolve()
    if dest_root_resolved != dest and dest_root_resolved not in dest.parents:
        raise RuntimeError(f"Refusing to extract path outside target dir: {rel}")

    dest.parent.mkdir(parents=True, exist_ok=True)
    with zf.open(info, "r") as src, open(dest, "wb") as dst:
        shutil.copyfileobj(src, dst)
    return dest


def extract_zip(zip_path: Path, out_dir: Path, expected_dir_name: str, force: bool = False) -> Path:
    expected_dir = out_dir / expected_dir_name
    if expected_dir.exists():
        if force:
            print(f"Removing existing folder: {expected_dir}")
            shutil.rmtree(expected_dir)
        else:
            print(f"Skip (already exists): {expected_dir}")
            return expected_dir

    with zipfile.ZipFile(zip_path) as zf:
        members = list(_iter_members(zf))
        if not members:
            raise RuntimeError(f"No extractable files found in {zip_path}")
        top_level = {name.split("/")[0] for _, name in members if name}
        has_expected = expected_dir_name in top_level

        if has_expected:
            print(f"Extracting {zip_path.name} into {out_dir} (keeping {expected_dir_name}/ ...)")
            for info, name in members:
                if not name.startswith(expected_dir_name + "/"):
                    continue
                _safe_extract_member(zf, info, out_dir, name)
        else:
            strip_components = 1 if len(top_level) == 1 else 0
            print(f"Extracting {zip_path.name} into {expected_dir} (strip_components={strip_components})")
            expected_dir.mkdir(parents=True, exist_ok=True)
            for info, name in members:
                parts = name.split("/")
                if strip_components and len(parts) > 1:
                    rel = "/".join(parts[strip_components:])
                else:
                    rel = name
                if not rel:
                    continue
                _safe_extract_member(zf, info, expected_dir, rel)

    return expected_dir


if DEV_ZIP is None and TEST_ZIP is None:
    raise FileNotFoundError(
        "No dataset zips found. Place the two ImageCLEF zip files in 0_dataset_prep/out "
        "or set AUTO_CONFIG['dev_zip'] / AUTO_CONFIG['test_zip']."
    )

if DEV_ZIP is not None:
    extract_zip(DEV_ZIP, OUT_DIR, DEV_NAME, force=FORCE)
else:
    print("Development zip not found; skipping.")

if TEST_ZIP is not None:
    extract_zip(TEST_ZIP, OUT_DIR, TEST_NAME, force=FORCE)
else:
    print("Testing zip not found; skipping.")


def _check_folder(folder: Path, required: list[str]):
    missing = [name for name in required if not (folder / name).exists()]
    if missing:
        print(f"Warning: {folder} is missing {missing}")
    else:
        print(f"OK: {folder}")


dev_dir = OUT_DIR / DEV_NAME
test_dir = OUT_DIR / TEST_NAME

if dev_dir.exists():
    _check_folder(dev_dir, ["images", "masks", "gt.json"])
else:
    print(f"Missing folder: {dev_dir}")

if test_dir.exists():
    _check_folder(test_dir, ["images"])
else:
    print(f"Missing folder: {test_dir}")


Extracting ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset.zip into /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/out (keeping ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset/ ...)
Extracting ImageCLEFmed-MEDVQA-GI-2023-Testing-Dataset.zip into /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/out (keeping ImageCLEFmed-MEDVQA-GI-2023-Testing-Dataset/ ...)
OK: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/out/ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset
OK: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/out/ImageCLEFmed-MEDVQA-GI-2023-Testing-Dataset
